# Example: Run a Risk Game

This needs work but shows how to bring in our game code, and run it in simple ways.

Thus just prints the log to the notebook, so it's not useful. See below for more useful examples.

In [ ]:
from nb_setup import risk_ai_game

opts = risk_ai_game.RiskAIGameOptions(
    agents=[
        risk_ai_game.AggressiveAgent(0, name="Aggressive-P0"),
        risk_ai_game.RandomAgent(1, aggression=0.3, name="Random-P1"),
    ],
    max_turns=1000,
    verbose=True,
)
risk_ai_game.run_game(opts)

# Example Analysis: Game Length

As an example analysis, we run 100 simple 1v1 games, and use a telemetry object that just collects
how long the game took to play out.

After collecting the data, we simply plot the game lengths as a histogram.

In [ ]:
# TurnCountCollector: run 100 games, histogram of game length
turn_counts = []
for _ in range(100):
    telemetry = risk_ai_game.TurnCountCollector()
    risk_ai_game.run_game(risk_ai_game.RiskAIGameOptions(
        agents=[
            risk_ai_game.AggressiveAgent(0, name="A"),
            risk_ai_game.RandomAgent(1, aggression=0.3, name="R"),
        ],
        max_turns=500,
        verbose=False,
        game_telemetry=telemetry,
    ))
    turn_counts.append(telemetry.turns)

import matplotlib.pyplot as plt
plt.hist(turn_counts, bins=20, edgecolor="black", alpha=0.7)
plt.xlabel("Turns to finish")
plt.ylabel("Count")
plt.title("Game length distribution (100 runs)")
plt.show()

# Example Analysis: Territory Counts

To show an example of a more complex telemetry object, we collect the territory count for each player at every phase of the game.

We then plot a simple graph showing how many each has. To make it interesting we add two agressive players, and two non-aggressive.

In [ ]:
telemetry = risk_ai_game.TerritoryCountCollector()
opts = risk_ai_game.RiskAIGameOptions(
    agents=[
        risk_ai_game.AggressiveAgent(0, name="Aggressive-P0"),
        risk_ai_game.RandomAgent(1, aggression=0.3, name="Random-P1"),
        risk_ai_game.AggressiveAgent(2, name="Aggressive-P2"),
        risk_ai_game.RandomAgent(3, aggression=0.4, name="Random-P3"),
    ],
    max_turns=200,
    verbose=False,
    game_telemetry=telemetry,
)
risk_ai_game.run_game(opts)
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
steps = range(len(telemetry.snapshots))
for p in range(len(opts.agents)):
    counts = [s[p] for s in telemetry.snapshots]
    ax.plot(steps, counts, label=f"Player {p}")
ax.set_xlabel("Game step")
ax.set_ylabel("Territories")
ax.legend()
ax.set_title("Territory count through the game")
plt.show()

# Custom Board Setup

To specify a specific board setup, pass a function to the `initial_board_setup` member of `RiskAIGameOptions`.

This example sets up a custom game, and shows the initial and final map state.

In [ ]:
from nb_setup import risk_ai_game, display_game_state

def setup_board(game):
    territories = game.board.all_territories()
    for i, t in enumerate(territories):
        t.owner = i % 2
        t.armies = 1
    game.armies_to_deploy = game.get_reinforcements(game.current_player)


telemetry = risk_ai_game.GameInitialFinalStates()
risk_ai_game.run_game(risk_ai_game.RiskAIGameOptions(
    agents=[
        risk_ai_game.AggressiveAgent(0, name="P0"),
        risk_ai_game.RandomAgent(1, aggression=0.3, name="P1"),
    ],
    initial_board_setup=setup_board,
    verbose=False,
    game_telemetry=telemetry,
))

display_game_state(telemetry.initial_state, width=600)
display_game_state(telemetry.final_state, width=600)